# 02 — Silver Layer: ETL Transformations

This notebook walks through the **Silver layer** ETL pipeline.

**Goal:** read raw Bronze data, apply cleansing and enrichment, validate quality,
and write clean Parquet to the Silver store.

**Silver contracts:**
- All critical columns are non-null.
- All prices are strictly positive.
- No duplicate `(ticker, trade_date)` pairs.
- `daily_return` = (close_t / close_{t-1}) − 1, per ticker.

In [ ]:
# Configuração do notebook da camada Silver (ETL).
import sys
sys.path.insert(0, "..")

import polars as pl
pl.Config.set_tbl_rows(20)

## 1. Read Bronze data

In [ ]:
# 1) Leitura da camada Bronze.
# Esta é a ENTRADA do pipeline Silver: dados brutos já persistidos.
# Se a Bronze estiver vazia, o pipeline aborta sem processar nada.
from d_processing.bronze.ingest import read_bronze

bronze_df = read_bronze(source="yahoo_finance")
print(f"Bronze rows: {len(bronze_df):,}  |  tickers: {bronze_df['ticker'].n_unique() if not bronze_df.is_empty() else 0}")
bronze_df.head(5)

## 2. Apply each transformation step individually

In [ ]:
# 2a) Primeiro passo do ETL: padronização de tipos.
# - Renomeia colunas para o padrão Silver (open → open_price, etc.).
# - Garante tipos numéricos consistentes (Float64) e datas como pl.Date.
# - Normaliza o ticker (strip + uppercase).
from d_processing.silver.transform import (
    cast_types,
    remove_nulls,
    remove_invalid_prices,
    deduplicate,
    calculate_daily_return,
)

step1 = cast_types(bronze_df)
print("After cast_types:")
print(step1.schema)
step1.head(3)

In [ ]:
# 2b) Aplicação sequencial das demais transformações da camada Silver.
# Cada etapa é uma função pura; isso facilita testes unitários (pasta z_tests).
step2 = remove_nulls(step1)             # remove linhas com nulos em colunas críticas
step3 = remove_invalid_prices(step2)    # remove linhas com preço <= 0
step4 = deduplicate(step3)              # mantém um registro por (ticker, trade_date)
step5 = calculate_daily_return(step4)   # calcula retorno diário = close_t / close_{t-1} - 1

# Auditoria simples: quantas linhas foram descartadas em cada etapa.
print(f"Rows after each step:")
print(f"  Bronze input  : {len(bronze_df):>7,}")
print(f"  cast_types    : {len(step1):>7,}")
print(f"  remove_nulls  : {len(step2):>7,}")
print(f"  invalid prices: {len(step3):>7,}")
print(f"  deduplicate   : {len(step4):>7,}")
print(f"  daily_return  : {len(step5):>7,}")

In [ ]:
# 2c) Amostra do resultado final do ETL com a métrica derivada `daily_return`.
step5.select(["ticker", "trade_date", "close_price", "daily_return"]).head(15)

## 3. Run quality checks

In [ ]:
# 3) Quality Gates: testes fail-fast sobre o DataFrame transformado.
# Garante o "contrato" da camada Silver:
#   - colunas-chave sem nulos
#   - preços positivos
#   - sem duplicatas (ticker, trade_date)
#   - sem datas no futuro
# Se qualquer asserção falhar, o pipeline aborta — não gravamos lixo na Silver.
from e_validation.quality_checks import run_silver_quality_checks

validated = run_silver_quality_checks(step5)
print("✔  All quality checks passed")

## 4. Write to Silver and verify

In [ ]:
# 4) Persistência: escreve a Silver e relê para confirmar.
# `transform_silver` encapsula todo o encadeamento de transformações acima.
from d_processing.silver.transform import write_silver, read_silver
from d_processing.silver.transform import transform_silver

silver_clean = transform_silver(bronze_df)
write_silver(silver_clean)

silver_df = read_silver()
print(f"Silver rows: {len(silver_df):,}")
silver_df.head(10)

## 5. Visualize daily returns

In [ ]:
# 5a) Visualização: série temporal dos retornos diários dos 5 ativos demo.
# Linha horizontal no zero serve como referência (acima = alta, abaixo = baixa).
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

top5 = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "ABEV3.SA"]
plot_df = silver_df.filter(pl.col("ticker").is_in(top5)).sort("trade_date").to_pandas()

fig = px.line(
    plot_df,
    x="trade_date",
    y="daily_return",
    color="ticker",
    title="Retorno Diário das Ações da B3 — Camada Silver",
    labels={
        "trade_date": "Data do Pregão",
        "daily_return": "Retorno Diário",
        "ticker": "Ativo",
    },
)
fig.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="0%", annotation_position="top left")
fig.update_layout(
    height=500,
    hovermode="x unified",
    legend=dict(title="Ativo", orientation="v", x=1.02, y=1),
    margin=dict(l=60, r=40, t=70, b=50),
)
fig.update_yaxes(tickformat=".2%")
fig.update_xaxes(tickformat="%d/%m/%Y")
fig.show()

In [ ]:
# 5b) Distribuição dos retornos diários (histograma).
# Sobreposição (`barmode="overlay"`) facilita comparar a dispersão entre ativos.
fig2 = px.histogram(
    plot_df.dropna(subset=["daily_return"]),
    x="daily_return",
    color="ticker",
    nbins=60,
    title="Distribuição dos Retornos Diários por Ativo",
    barmode="overlay",
    opacity=0.55,
    labels={
        "daily_return": "Retorno Diário",
        "ticker": "Ativo",
        "count": "Frequência",
    },
)
fig2.update_layout(
    height=480,
    legend=dict(title="Ativo"),
    margin=dict(l=60, r=40, t=70, b=50),
    yaxis_title="Frequência",
)
fig2.update_xaxes(tickformat=".1%")
fig2.show()

## 6. Run full Silver pipeline

In [ ]:
# 6) Execução do pipeline Silver via classe (mesmo objeto usado pelo Airflow).
# `SilverPipeline.run()` faz: read_bronze → transform_silver → quality_checks → write_silver.
from f_pipelines.silver_pipeline import SilverPipeline

result = SilverPipeline().run()
print(f"SilverPipeline produced {len(result):,} clean rows")